Google Ads & GA4 Automated Data Pipeline

**Stack:** Python · Google Ads API · GA4 API · SQL Server · Power BI  
**Purpose:** End-to-end automated marketing analytics pipeline — extracts data from Google Ads and GA4 APIs, cleans and standardises it, and loads it into SQL Server to power live Power BI dashboards.

---

## Pipeline Architecture

```
Google Ads API ──┐
                 ├──▶ Python (Extraction) ──▶ Raw CSVs ──▶ Python (Cleaning) ──▶ Clean CSVs ──▶ SQL Server ──▶ Power BI
GA4 API ─────────┘
```

## 1. Setup & Library Verification

In [2]:
# Core libraries
import pandas as pd
import numpy as np
from datetime import datetime
import os
import glob

# Google Ads API
from google.ads.googleads.client import GoogleAdsClient
from google.ads.googleads.errors import GoogleAdsException

# GA4 API
from google.oauth2.credentials import Credentials
from google.analytics.data_v1beta import BetaAnalyticsDataClient
from google.analytics.data_v1beta.types import (
    RunReportRequest, DateRange, Metric, Dimension
)

# SQL Server
from sqlalchemy import create_engine, text

print("All libraries loaded successfully")

All libraries loaded successfully


## 2. Google Ads API Extraction

In [5]:
# ── CONFIG ─────────────────────────────────────────────────────────────────────

import yaml

YAML_PATH      = r'C:\Pipeline\Credentials\google-ads.yaml'
RAW_OUTPUT_DIR = r'C:\Pipeline\Data\Raw'

with open(YAML_PATH) as f:
    ads_creds = yaml.safe_load(f)

CUSTOMER_ID = str(ads_creds["login_customer_id"])

START_DATE = "2023-01-01"
END_DATE   = datetime.today().strftime("%Y-%m-%d")

os.makedirs(RAW_OUTPUT_DIR, exist_ok=True)

client     = GoogleAdsClient.load_from_storage(YAML_PATH)
ga_service = client.get_service("GoogleAdsService")
timestamp  = datetime.now().strftime("%Y%m%d_%H%M")

print("=" * 60)
print(f"Google Ads Extraction — {timestamp}")
print(f"Date range: {START_DATE} → {END_DATE}")
print("=" * 60)

# ── HELPERS ────────────────────────────────────────────────────────────────────

def run_query(query, label):
    try:
        response = ga_service.search(customer_id=CUSTOMER_ID, query=query)
        rows = list(response)
        print(f"{label} — {len(rows)} rows")
        return rows
    except GoogleAdsException as ex:
        print(f"{label} — FAILED: {ex.error.code().name}")
        for error in ex.failure.errors:
            print(f"     {error.message}")
        return []

def save_raw(df, name):
    path = os.path.join(RAW_OUTPUT_DIR, f"{name}_{timestamp}.csv")
    df.to_csv(path, index=False)
    print(f"Saved → {path}")
    return path

# ── 1. CAMPAIGNS ───────────────────────────────────────────────────────────────

print(f"\n[1/4] Campaigns")
rows = run_query(f"""
    SELECT
        campaign.name,
        campaign.advertising_channel_type,
        campaign.status,
        metrics.impressions,
        metrics.clicks,
        metrics.cost_micros,
        metrics.conversions,
        metrics.conversions_value,
        metrics.all_conversions,
        metrics.all_conversions_value,
        metrics.ctr,
        segments.date
    FROM campaign
    WHERE segments.date BETWEEN '{START_DATE}' AND '{END_DATE}'
      AND campaign.status != 'REMOVED'
    ORDER BY segments.date DESC
""", "Campaigns")

campaigns_df = pd.DataFrame([{
    "date":                  r.segments.date,
    "campaign":              r.campaign.name,
    "campaign_type":         r.campaign.advertising_channel_type.name,
    "status":                r.campaign.status.name,
    "impressions":           r.metrics.impressions,
    "clicks":                r.metrics.clicks,
    "cost":                  r.metrics.cost_micros / 1_000_000,
    "conversions":           r.metrics.conversions,
    "conversions_value":     r.metrics.conversions_value,
    "all_conversions":       r.metrics.all_conversions,
    "all_conversions_value": r.metrics.all_conversions_value,
    "ctr":                   r.metrics.ctr,
} for r in rows])

save_raw(campaigns_df, "campaigns_raw")

# ── 2. AD GROUPS ───────────────────────────────────────────────────────────────

print(f"\n[2/4] Ad Groups")
rows = run_query(f"""
    SELECT
        campaign.name,
        campaign.advertising_channel_type,
        ad_group.name,
        metrics.impressions,
        metrics.clicks,
        metrics.cost_micros,
        metrics.conversions,
        metrics.conversions_value,
        metrics.all_conversions,
        metrics.all_conversions_value,
        metrics.ctr,
        segments.date
    FROM ad_group
    WHERE segments.date BETWEEN '{START_DATE}' AND '{END_DATE}'
      AND campaign.status != 'REMOVED'
      AND ad_group.status != 'REMOVED'
    ORDER BY segments.date DESC
""", "Ad Groups")

ad_groups_df = pd.DataFrame([{
    "date":                  r.segments.date,
    "campaign":              r.campaign.name,
    "campaign_type":         r.campaign.advertising_channel_type.name,
    "ad_group":              r.ad_group.name,
    "impressions":           r.metrics.impressions,
    "clicks":                r.metrics.clicks,
    "cost":                  r.metrics.cost_micros / 1_000_000,
    "conversions":           r.metrics.conversions,
    "conversions_value":     r.metrics.conversions_value,
    "all_conversions":       r.metrics.all_conversions,
    "all_conversions_value": r.metrics.all_conversions_value,
    "ctr":                   r.metrics.ctr,
} for r in rows])

save_raw(ad_groups_df, "ad_groups_raw")

# ── 3. KEYWORDS ────────────────────────────────────────────────────────────────

print(f"\n[3/4] Keywords")
rows = run_query(f"""
    SELECT
        campaign.name,
        ad_group.name,
        ad_group_criterion.keyword.text,
        ad_group_criterion.keyword.match_type,
        metrics.impressions,
        metrics.clicks,
        metrics.cost_micros,
        metrics.conversions,
        metrics.conversions_value,
        metrics.all_conversions_value,
        metrics.ctr,
        segments.date
    FROM keyword_view
    WHERE segments.date BETWEEN '{START_DATE}' AND '{END_DATE}'
      AND campaign.status != 'REMOVED'
      AND ad_group.status != 'REMOVED'
      AND ad_group_criterion.status != 'REMOVED'
    ORDER BY segments.date DESC
""", "Keywords")

keywords_df = pd.DataFrame([{
    "date":                  r.segments.date,
    "campaign":              r.campaign.name,
    "ad_group":              r.ad_group.name,
    "keyword":               r.ad_group_criterion.keyword.text,
    "match_type":            r.ad_group_criterion.keyword.match_type.name,
    "impressions":           r.metrics.impressions,
    "clicks":                r.metrics.clicks,
    "cost":                  r.metrics.cost_micros / 1_000_000,
    "conversions":           r.metrics.conversions,
    "conversions_value":     r.metrics.conversions_value,
    "all_conversions_value": r.metrics.all_conversions_value,
    "ctr":                   r.metrics.ctr,
} for r in rows])

save_raw(keywords_df, "keywords_raw")

# ── 4. CONVERSION ATTRIBUTION ──────────────────────────────────────────────────

print(f"\n[4/4] Conversion Attribution")
rows = run_query(f"""
    SELECT
        campaign.name,
        metrics.all_conversions,
        metrics.conversions_value,
        segments.conversion_attribution_event_type,
        segments.date
    FROM campaign
    WHERE segments.date BETWEEN '{START_DATE}' AND '{END_DATE}'
      AND campaign.status != 'REMOVED'
    ORDER BY segments.date DESC
""", "Attribution")

attribution_df = pd.DataFrame([{
    "date":              r.segments.date,
    "campaign":          r.campaign.name,
    "all_conversions":   r.metrics.all_conversions,
    "conversions_value": r.metrics.conversions_value,
    "attribution_event": r.segments.conversion_attribution_event_type.name,
} for r in rows])

save_raw(attribution_df, "attribution_raw")

# ── SUMMARY ────────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("EXTRACTION COMPLETE")
print("=" * 60)
print(f"  Campaigns:   {len(campaigns_df):,} rows")
print(f"  Ad Groups:   {len(ad_groups_df):,} rows")
print(f"  Keywords:    {len(keywords_df):,} rows")
print(f"  Attribution: {len(attribution_df):,} rows")
print(f"\n  Raw files saved to {RAW_OUTPUT_DIR}")

Google Ads Extraction — 20260330_2149
Date range: 2023-01-01 → 2026-03-30

[1/4] Campaigns
  ✅ Campaigns — 8429 rows
  💾 Saved → C:\Pipeline\Data\Raw\campaigns_raw_20260330_2149.csv

[2/4] Ad Groups
  ✅ Ad Groups — 9585 rows
  💾 Saved → C:\Pipeline\Data\Raw\ad_groups_raw_20260330_2149.csv

[3/4] Keywords
  ✅ Keywords — 28090 rows
  💾 Saved → C:\Pipeline\Data\Raw\keywords_raw_20260330_2149.csv

[4/4] Conversion Attribution
  ✅ Attribution — 1383 rows
  💾 Saved → C:\Pipeline\Data\Raw\attribution_raw_20260330_2149.csv

EXTRACTION COMPLETE
  Campaigns:   8,429 rows
  Ad Groups:   9,585 rows
  Keywords:    28,090 rows
  Attribution: 1,383 rows

  Raw files saved to C:\Pipeline\Data\Raw


## 3. GA4 API Extraction

Extracts 3 tables from the GA4 Data API:
1. **Traffic Acquisition** — sessions by channel group (daily)
2. **User Acquisition** — new/returning users by first user channel (daily)
3. **Monthly Performance** — aggregated engagement metrics by month

In [7]:
# ── CONFIG ─────────────────────────────────────────────────────────────────────

RAW_OUTPUT_DIR = r'C:\Pipeline\Data\Raw'

import yaml
with open(r'C:\Pipeline\Credentials\google-ads.yaml') as f:
    ads_creds = yaml.safe_load(f)

GA4_PROPERTY_ID = str(ads_creds["ga4_property_id"])

creds = Credentials(
    token=None,
    refresh_token=ads_creds["refresh_token"],
    client_id=ads_creds["client_id"],
    client_secret=ads_creds["client_secret"],
    token_uri="https://oauth2.googleapis.com/token"
)

START_DATE = "2023-01-01"
END_DATE   = "today"

os.makedirs(RAW_OUTPUT_DIR, exist_ok=True)
client    = BetaAnalyticsDataClient(credentials=creds)
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

print("=" * 60)
print(f"GA4 Extraction — {timestamp}")
print(f"Property:   {GA4_PROPERTY_ID}")
print(f"Date range: {START_DATE} -> {END_DATE}")
print("=" * 60)

# ── HELPER ─────────────────────────────────────────────────────────────────────

def parse_response(response):
    """Convert GA4 API response into a flat list of dicts."""
    dim_headers = [h.name for h in response.dimension_headers]
    met_headers = [h.name for h in response.metric_headers]
    rows = []
    for row in response.rows:
        record = {}
        for i, val in enumerate(row.dimension_values):
            record[dim_headers[i]] = val.value
        for i, val in enumerate(row.metric_values):
            record[met_headers[i]] = val.value
        rows.append(record)
    return rows

def save_raw(df, name):
    path = os.path.join(RAW_OUTPUT_DIR, f"{name}_{timestamp}.csv")
    df.to_csv(path, index=False)
    print(f"  Saved -> {path}")
    return path

# ── 1. TRAFFIC ACQUISITION ─────────────────────────────────────────────────────

print(f"\n[1/3] Traffic Acquisition - Sessions")

response = client.run_report(RunReportRequest(
    property=f"properties/{GA4_PROPERTY_ID}",
    date_ranges=[DateRange(start_date=START_DATE, end_date=END_DATE)],
    dimensions=[
        Dimension(name="sessionPrimaryChannelGroup"),
        Dimension(name="date"),
    ],
    metrics=[
        Metric(name="sessions"),
        Metric(name="engagedSessions"),
        Metric(name="engagementRate"),
        Metric(name="eventCount"),
        Metric(name="eventsPerSession"),
        Metric(name="keyEvents"),
        Metric(name="averageSessionDuration"),
    ]
))

traffic_df = pd.DataFrame(parse_response(response))
traffic_df.columns = [
    "session_channel", "date",
    "sessions", "engaged_sessions", "engagement_rate",
    "event_count", "events_per_session", "key_events",
    "avg_session_duration"
]
print(f"  Traffic Acquisition — {len(traffic_df)} rows")
save_raw(traffic_df, "ga4_traffic_acquisition_raw")

# ── 2. USER ACQUISITION ────────────────────────────────────────────────────────

print(f"\n[2/3] User Acquisition - First User")

response = client.run_report(RunReportRequest(
    property=f"properties/{GA4_PROPERTY_ID}",
    date_ranges=[DateRange(start_date=START_DATE, end_date=END_DATE)],
    dimensions=[
        Dimension(name="firstUserPrimaryChannelGroup"),
        Dimension(name="date"),
    ],
    metrics=[
        Metric(name="newUsers"),
        Metric(name="totalUsers"),
        Metric(name="engagedSessions"),
        Metric(name="keyEvents"),
        Metric(name="eventCount"),
    ]
))

users_df = pd.DataFrame(parse_response(response))
users_df.columns = [
    "first_user_channel", "date",
    "new_users", "total_users", "engaged_sessions_per_user",
    "key_events", "event_count"
]
print(f"  User Acquisition — {len(users_df)} rows")
save_raw(users_df, "ga4_user_acquisition_raw")

# ── 3. MONTHLY PERFORMANCE ─────────────────────────────────────────────────────

print(f"\n[3/3] Monthly Performance")

response = client.run_report(RunReportRequest(
    property=f"properties/{GA4_PROPERTY_ID}",
    date_ranges=[DateRange(start_date=START_DATE, end_date=END_DATE)],
    dimensions=[
        Dimension(name="month"),
        Dimension(name="year"),
    ],
    metrics=[
        Metric(name="keyEvents"),
        Metric(name="eventCount"),
        Metric(name="engagementRate"),
        Metric(name="averageSessionDuration"),
        Metric(name="sessionKeyEventRate"),
    ]
))

monthly_df = pd.DataFrame(parse_response(response))
monthly_df.columns = [
    "month", "year",
    "key_events", "event_count", "engagement_rate",
    "avg_session_duration", "session_key_event_rate"
]
print(f"  Monthly Performance — {len(monthly_df)} rows")
save_raw(monthly_df, "ga4_monthly_performance_raw")

# ── SUMMARY ────────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("EXTRACTION COMPLETE")
print("=" * 60)
print(f"  Traffic Acquisition: {len(traffic_df)} rows")
print(f"  User Acquisition:    {len(users_df)} rows")
print(f"  Monthly Performance: {len(monthly_df)} rows")
print(f"\n  Raw files saved to {RAW_OUTPUT_DIR}")

GA4 Extraction — 20260330_2152
Property:   349322104
Date range: 2023-01-01 -> today

[1/3] Traffic Acquisition - Sessions
  Traffic Acquisition — 9256 rows
  Saved -> C:\Pipeline\Data\Raw\ga4_traffic_acquisition_raw_20260330_2152.csv

[2/3] User Acquisition - First User
  User Acquisition — 9257 rows
  Saved -> C:\Pipeline\Data\Raw\ga4_user_acquisition_raw_20260330_2152.csv

[3/3] Monthly Performance
  Monthly Performance — 38 rows
  Saved -> C:\Pipeline\Data\Raw\ga4_monthly_performance_raw_20260330_2152.csv

EXTRACTION COMPLETE
  Traffic Acquisition: 9256 rows
  User Acquisition:    9257 rows
  Monthly Performance: 38 rows

  Raw files saved to C:\Pipeline\Data\Raw


## 4. Data Cleaning & Standardisation

In [8]:
# ── CONFIG ─────────────────────────────────────────────────────────────────────

RAW_DIR   = r'C:\Pipeline\Data\Raw'
CLEAN_DIR = r'C:\Pipeline\Data\Clean'

os.makedirs(CLEAN_DIR, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

print("=" * 60)
print(f"Cleaning Script — {timestamp}")
print("=" * 60)

# ── HELPER: LOAD MOST RECENT FILE ──────────────────────────────────────────────

def load_latest(pattern, skiprows=0):
    files = glob.glob(os.path.join(RAW_DIR, pattern))
    if not files:
        raise FileNotFoundError(f"No files found matching: {pattern}")
    latest = max(files, key=os.path.getmtime)
    print(f" Loading: {os.path.basename(latest)}")
    return pd.read_csv(latest, skiprows=skiprows)

# ── LOAD RAW FILES ─────────────────────────────────────────────────────────────

print("\n[1/4] Loading raw files...")
campaigns_raw   = load_latest("campaigns_raw_*.csv")
ad_groups_raw   = load_latest("ad_groups_raw_*.csv")
keywords_raw    = load_latest("keywords_raw_*.csv")
attribution_raw = load_latest('Conversion_attribution_analysis*.csv', skiprows=9)
traffic_raw     = load_latest("ga4_traffic_acquisition_raw_*.csv")
users_raw       = load_latest("ga4_user_acquisition_raw_*.csv")
monthly_raw     = load_latest("ga4_monthly_performance_raw_*.csv")
print("All 7 files loaded")

# ── CAMPAIGN NAME STANDARDISATION ──────────────────────────────────────────────
# Maps raw Google Ads campaign names to clean standardised labels

campaign_mapping = {
    'DbB: Brasil': 'Search_Brand_BR',
    'DB | EUA - Search': 'Search_Brand_USA',
    'DB | Espanha - Search': 'Search_Brand_ESP',
    'DB | Brasil - Search': 'Search_Brand_BR_2',
    'DB | Brasil - Remarketing': 'Remarketing_BR',
    'DB | Espanha - Remarketing': 'Remarketing_ESP',
    'Sales_UK_IE (EN Marca Liberdade)': 'Sales_UK_IE',
    'Sales_USA_CA': 'Sales_USA_CA',
    'Sales FR_BE_CH (FR Marca Liberdade)': 'Sales_FR_BE_CH',
    'BA_CityBreak_UK_IE_DE_NL': 'Brand_CityBreak_Europe',
    'Programa Dia dos Namorados': 'Seasonal_Valentines_PT',
    'Restaurante Open PT': 'Restaurant_PT',
    'Brand Awareness FR': 'Brand_Awareness_FR',
    'Sales_DE_CH_BE_NL': 'Sales_DE_CH_BE_NL',
    'Brand Awareness - MICE PT': 'Brand_MICE_PT',
    'Brand_Awareness_PT': 'Brand_Awareness_PT',
    'Brand Awareness_USA': 'Brand_Awareness_USA',
    'Brand_Discover_FR': 'Brand_Discovery_FR',
    'Brand_Discover_ESP': 'Brand_Discovery_ESP',
    'Sales_ESP': 'Sales_ESP',
    'Brand_Discovery_DE_NL_CH_BE': 'Brand_Discovery_DE_NL_CH_BE',
    'BrandDiscovery_Display_ESP': 'Brand_Discovery_Display_ESP',
    'Brand Awareness_ESP': 'Brand_Awareness_ESP',
    'Brand_Discovery_Europe': 'Brand_Discovery_Europe',
    'BA_Discovery_PT': 'Brand_Discovery_PT',
    'Sales PT (PT Marca Liberdade)': 'Sales_PT',
    'Brand_Discover_USA': 'Brand_Discovery_USA',
    'Brand_Discovery_UK': 'Brand_Discovery_UK',
    'Brand_Discovery_BR': 'Brand_Discovery_BR',
    'Restaurante Open_ENG': 'Restaurant_ENG',
    'BADC_Semana_Santa_ESP': 'Seasonal_Easter_ESP',
    'BAS_Passagem_Ano_PT': 'Seasonal_NewYear_PT',
    'BAD_Passagem_Ano_PT': 'Seasonal_NewYear_PT_2',
    'Remarketing ENG': 'Remarketing_ENG',
    'RMKTG_Display_ENG': 'Remarketing_Display_ENG',
    'Brand_Awareness_BR': 'Brand_Awareness_BR',
    'Brand_Demand_Gen_USA': 'DemandGen_USA',
    'BAS_Hotels_Lisbon_Europe_Green': 'Brand_Sustainability_Europe',
    'BAD_DemandGen_Scandinavia': 'DemandGen_Scandinavia',
    'PerformMax_ESP': 'PerformanceMax_ESP',
    'BAS_Hotels_Lisbon_Exclude Green': 'Brand_Hotels_Europe',
    'Republica | Inspira Santos - Search - Marca': 'Search_Brand_Hotel2',
    'Republica | Inspira Santos - Search - Localização': 'Search_Location_Hotel2',
    'Republica | Inspira Santos - Pmax - EN - United Kingdom': 'PMax_Hotel2_UK',
    'Republica | Inspira Santos - Pmax - EN - New York': 'PMax_Hotel2_USA',
    'Republica | Inspira Liberdade - Pmax - EN - United Kingdom': 'PMax_Hotel1_UK',
    'Republica | Inspira Liberdade - Pmax - EN - New York': 'PMax_Hotel1_USA',
}

# ── CLEAN: CAMPAIGNS ───────────────────────────────────────────────────────────

print("\n[2/4] Cleaning Google Ads tables...")

df = campaigns_raw.copy()
df['date']     = pd.to_datetime(df['date'])
df['Month']    = df['date'].dt.to_period('M').dt.to_timestamp()
df['Year']     = df['date'].dt.year
df['campaign'] = df['campaign'].replace(campaign_mapping)

agg = df.groupby(['Month', 'Year', 'campaign_type', 'campaign']).agg(
    Impressions    = ('impressions', 'sum'),
    Clicks         = ('clicks', 'sum'),
    Conversions    = ('conversions', 'sum'),
    All_conv_value = ('all_conversions_value', 'sum'),
    Cost           = ('cost', 'sum'),
).reset_index()

agg['Value_per_conv']    = np.where(agg['Conversions'] > 0, agg['All_conv_value'] / agg['Conversions'], 0)
agg['All_conv_rate']     = np.where(agg['Clicks'] > 0, agg['Conversions'] / agg['Clicks'] * 100, 0)
agg['CTR']               = np.where(agg['Impressions'] > 0, agg['Clicks'] / agg['Impressions'] * 100, 0)
agg['Search_impr_share'] = np.nan  # not available via API

campaigns = agg.rename(columns={'campaign_type': 'Campaign_type', 'campaign': 'Campaign'})[
    ['Month', 'Year', 'Campaign_type', 'Campaign', 'Impressions', 'Clicks',
     'Conversions', 'Value_per_conv', 'All_conv_rate', 'All_conv_value',
     'CTR', 'Search_impr_share', 'Cost']]

print(f" campaigns_clean: {len(campaigns):,} rows")

# ── CLEAN: AD GROUPS ───────────────────────────────────────────────────────────

df = ad_groups_raw.copy()
df['date']     = pd.to_datetime(df['date'])
df['Month']    = df['date'].dt.to_period('M').dt.to_timestamp()
df['Year']     = df['date'].dt.year
df['campaign'] = df['campaign'].replace(campaign_mapping)

agg = df.groupby(['Month', 'Year', 'campaign_type', 'campaign', 'ad_group']).agg(
    Impressions    = ('impressions', 'sum'),
    Clicks         = ('clicks', 'sum'),
    Conversions    = ('conversions', 'sum'),
    All_conv_value = ('all_conversions_value', 'sum'),
    Cost           = ('cost', 'sum'),
).reset_index()

agg['Value_per_conv']    = np.where(agg['Conversions'] > 0, agg['All_conv_value'] / agg['Conversions'], 0)
agg['All_conv_rate']     = np.where(agg['Clicks'] > 0, agg['Conversions'] / agg['Clicks'] * 100, 0)
agg['CTR']               = np.where(agg['Impressions'] > 0, agg['Clicks'] / agg['Impressions'] * 100, 0)
agg['Search_impr_share'] = np.nan

ad_groups = agg.rename(columns={
    'campaign_type': 'Campaign_type', 'campaign': 'Campaign', 'ad_group': 'Ad_group'})[
    ['Month', 'Year', 'Campaign_type', 'Campaign', 'Ad_group', 'Impressions',
     'Clicks', 'Conversions', 'Value_per_conv', 'All_conv_rate', 'All_conv_value',
     'CTR', 'Search_impr_share', 'Cost']]

print(f" ad_groups_clean: {len(ad_groups):,} rows")

# ── CLEAN: KEYWORDS ────────────────────────────────────────────────────────────

df = keywords_raw.copy()
df['date']     = pd.to_datetime(df['date'])
df['Year']     = df['date'].dt.year
df['Month']    = df['date'].dt.strftime('%B %Y')  # e.g. "January 2023"
df['campaign'] = df['campaign'].replace(campaign_mapping)

agg = df.groupby(['keyword', 'match_type', 'campaign', 'ad_group', 'Year', 'Month']).agg(
    Cost           = ('cost', 'sum'),
    Impressions    = ('impressions', 'sum'),
    Clicks         = ('clicks', 'sum'),
    Conversions    = ('conversions', 'sum'),
    All_conv_value = ('all_conversions_value', 'sum'),
).reset_index()

agg['CTR']               = np.where(agg['Impressions'] > 0, agg['Clicks'] / agg['Impressions'] * 100, 0)
agg['Value_per_conv']    = np.where(agg['Conversions'] > 0, agg['All_conv_value'] / agg['Conversions'], 0)
agg['Search_impr_share'] = np.nan
agg['Quality_score']     = np.nan  # not available via API

keywords = agg.rename(columns={
    'keyword': 'Keyword', 'match_type': 'Match_type',
    'campaign': 'Campaign', 'ad_group': 'Ad_group'})[
    ['Keyword', 'Match_type', 'Campaign', 'Ad_group', 'Year', 'Quality_score',
     'Month', 'Cost', 'Impressions', 'Clicks', 'CTR', 'Conversions',
     'All_conv_value', 'Value_per_conv', 'Search_impr_share']]

print(f" keywords_clean: {len(keywords):,} rows")

# ── CLEAN: ATTRIBUTION (manual GA4 export) ─────────────────────────────────────

df = attribution_raw.copy()

if df.columns[0].startswith('#') or 'Primary channel group' not in df.columns[0]:
    df = pd.read_csv(
        max(glob.glob(os.path.join(RAW_DIR, 'Conversion_attribution_analysis*.csv')),
            key=os.path.getmtime),
        skiprows=9
    )

df.columns = [
    'Primary_channel_group_Default_channel_group',
    'Single_touch_point_credit_by_int_time',
    'Early_touch_point_credit_by_int_time',
    'Mid_touch_point_credit_by_int_time',
    'Late_touch_point_credit_by_int_time',
    'All_conversions',
    'Total_revenue_by_int_time',
    'Ads_cost'
]

for col in df.columns[1:]:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

attribution = df.copy()
print(f"Conversion_attribution_analysis: {len(attribution):,} rows")

# ── CLEAN: GA4 TRAFFIC ─────────────────────────────────────────────────────────

print("\n[3/4] Cleaning GA4 tables...")

df = traffic_raw.copy()
for col in ['sessions', 'engaged_sessions', 'engagement_rate',
            'events_per_session', 'event_count', 'key_events', 'avg_session_duration']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

agg = df.groupby('session_channel').agg(
    Sessions           = ('sessions', 'sum'),
    Engaged_sessions   = ('engaged_sessions', 'sum'),
    Engagement_rate    = ('engagement_rate', 'mean'),
    Average_engagement_time_per_session = ('avg_session_duration', 'mean'),
    Events_per_session = ('events_per_session', 'mean'),
    Event_count        = ('event_count', 'sum'),
    Key_events         = ('key_events', 'sum'),
    Session_key_event_rate = ('engagement_rate', 'mean'),
).reset_index()

agg['Total_revenue'] = 0

traffic = agg.rename(columns={
    'session_channel': 'Session_primary_channel_group_Default_channel_group'})[
    ['Session_primary_channel_group_Default_channel_group',
     'Sessions', 'Engaged_sessions', 'Engagement_rate',
     'Average_engagement_time_per_session', 'Events_per_session',
     'Event_count', 'Key_events', 'Session_key_event_rate', 'Total_revenue']]

print(f" Traffic_acquisition: {len(traffic):,} rows")

# ── CLEAN: GA4 USERS ───────────────────────────────────────────────────────────

df = users_raw.copy()
for col in ['new_users', 'total_users', 'engaged_sessions_per_user', 'key_events', 'event_count']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

agg = df.groupby('first_user_channel').agg(
    Total_users      = ('total_users', 'sum'),
    New_users        = ('new_users', 'sum'),
    Engaged_sessions_per_active_user = ('engaged_sessions_per_user', 'mean'),
    Event_count      = ('event_count', 'sum'),
    Key_events       = ('key_events', 'sum'),
).reset_index()

agg['Returning_users']                         = agg['Total_users'] - agg['New_users']
agg['Average_engagement_time_per_active_user'] = 0
agg['User_key_event_rate']                     = np.where(
    agg['Total_users'] > 0, agg['Key_events'] / agg['Total_users'], 0)

users = agg.rename(columns={
    'first_user_channel': 'First_user_primary_channel_group_Default_channel_group'})[
    ['First_user_primary_channel_group_Default_channel_group',
     'Total_users', 'New_users', 'Returning_users',
     'Average_engagement_time_per_active_user',
     'Engaged_sessions_per_active_user', 'Event_count',
     'Key_events', 'User_key_event_rate']]

print(f" User_acquisition: {len(users):,} rows")

# ── CLEAN: GA4 MONTHLY ─────────────────────────────────────────────────────────

df = monthly_raw.copy()
df['Month'] = pd.to_numeric(df['month'], errors='coerce').astype(int)
df['Year']  = pd.to_numeric(df['year'], errors='coerce').astype(int)

traffic_monthly        = traffic_raw.copy()
traffic_monthly['date']  = pd.to_datetime(traffic_monthly['date'], format='%Y%m%d')
traffic_monthly['Month'] = traffic_monthly['date'].dt.month
traffic_monthly['Year']  = traffic_monthly['date'].dt.year

for col in ['sessions', 'engaged_sessions', 'events_per_session', 'avg_session_duration']:
    traffic_monthly[col] = pd.to_numeric(traffic_monthly[col], errors='coerce').fillna(0)

traffic_agg = traffic_monthly.groupby(['Year', 'Month']).agg(
    Sessions           = ('sessions', 'sum'),
    Engaged_sessions   = ('engaged_sessions', 'sum'),
    Events_per_session = ('events_per_session', 'mean'),
    Average_engagement_time_per_session = ('avg_session_duration', 'mean'),
).reset_index()

monthly = df.merge(traffic_agg, on=['Year', 'Month'], how='left')
monthly['Engagement_rate']        = pd.to_numeric(monthly['engagement_rate'], errors='coerce').fillna(0)
monthly['Event_count']            = pd.to_numeric(monthly['event_count'], errors='coerce').fillna(0).astype(int)
monthly['Key_events']             = pd.to_numeric(monthly['key_events'], errors='coerce').fillna(0).astype(int)
monthly['Session_key_event_rate'] = pd.to_numeric(monthly['session_key_event_rate'], errors='coerce').fillna(0)
monthly['Total_revenue']          = 0

monthly = monthly[[
    'Month', 'Year', 'Sessions', 'Engaged_sessions', 'Engagement_rate',
    'Average_engagement_time_per_session', 'Events_per_session',
    'Event_count', 'Key_events', 'Session_key_event_rate', 'Total_revenue'
]].sort_values(['Year', 'Month']).reset_index(drop=True)

print(f" Month final: {len(monthly):,} rows")

# ── SAVE CLEAN FILES ───────────────────────────────────────────────────────────

print("\n[4/4] Saving clean files...")

def save_clean(df, name):
    path = os.path.join(CLEAN_DIR, f"{name}_{timestamp}.csv")
    df.to_csv(path, index=False)
    print(f" {name}: {len(df):,} rows → {path}")

save_clean(campaigns,   "campaigns_clean")
save_clean(ad_groups,   "ad_groups_clean")
save_clean(keywords,    "keywords_clean")
save_clean(attribution, "attribution_clean")
save_clean(traffic,     "ga4_traffic_clean")
save_clean(users,       "ga4_users_clean")
save_clean(monthly,     "ga4_monthly_clean")

print("\n" + "=" * 60)
print("CLEANING COMPLETE")
print("=" * 60)
print(f"  campaigns_clean:        {len(campaigns):,} rows")
print(f"  ad_groups_clean:        {len(ad_groups):,} rows")
print(f"  keywords_clean:         {len(keywords):,} rows")
print(f"  attribution_analysis:   {len(attribution):,} rows")
print(f"  ga4_traffic:            {len(traffic):,} rows")
print(f"  ga4_users:              {len(users):,} rows")
print(f"  ga4_monthly:            {len(monthly):,} rows")
print(f"\n  Clean files saved to {CLEAN_DIR}")

Cleaning Script — 20260330_2153

[1/4] Loading raw files...
 Loading: campaigns_raw_20260330_2149.csv
 Loading: ad_groups_raw_20260330_2149.csv
 Loading: keywords_raw_20260330_2149.csv
 Loading: Conversion_attribution_analysis.csv
 Loading: ga4_traffic_acquisition_raw_20260330_2152.csv
 Loading: ga4_user_acquisition_raw_20260330_2152.csv
 Loading: ga4_monthly_performance_raw_20260330_2152.csv
All 7 files loaded

[2/4] Cleaning Google Ads tables...
 campaigns_clean: 356 rows
 ad_groups_clean: 447 rows
 keywords_clean: 1,786 rows
  ✅ Conversion_attribution_analysis: 9 rows

[3/4] Cleaning GA4 tables...
 Traffic_acquisition: 11 rows
 User_acquisition: 11 rows
 Month final: 38 rows

[4/4] Saving clean files...
 campaigns_clean: 356 rows → C:\Pipeline\Data\Clean\campaigns_clean_20260330_2153.csv
 ad_groups_clean: 447 rows → C:\Pipeline\Data\Clean\ad_groups_clean_20260330_2153.csv
 keywords_clean: 1,786 rows → C:\Pipeline\Data\Clean\keywords_clean_20260330_2153.csv
 attribution_clean: 9 rows

## 5. SQL Server Load

In [9]:
# ── CONFIG ─────────────────────────────────────────────────────────────────────

CLEAN_DIR    = r'C:\Pipeline\Data\Clean'
SQL_SERVER   = r'DUARTEPC\SQLEXPRESS'
SQL_DATABASE = 'google_ads_analysis'

timestamp = datetime.now().strftime("%Y%m%d_%H%M")

print("=" * 60)
print(f"SQL Server Load — {timestamp}")
print(f"Server:   {SQL_SERVER}")
print(f"Database: {SQL_DATABASE}")
print("=" * 60)

# ── CONNECTION — Windows Authentication ────────────────────────────────────────

connection_string = (
    "mssql+pyodbc:///?odbc_connect="
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={SQL_SERVER};"
    f"DATABASE={SQL_DATABASE};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(connection_string, fast_executemany=True)

with engine.connect() as conn:
    conn.execute(text("SELECT 1"))
print(" Connected to SQL Server\n")

# ── HELPERS ────────────────────────────────────────────────────────────────────

def load_latest_clean(pattern):
    files = glob.glob(os.path.join(CLEAN_DIR, pattern))
    if not files:
        raise FileNotFoundError(f"No files found: {pattern}")
    latest = max(files, key=os.path.getmtime)
    print(f" {os.path.basename(latest)}")
    return pd.read_csv(latest)

def append_after_max_month(df, table_name, date_col="Month"):
    with engine.connect() as conn:
        exists = conn.execute(text(f"SELECT COUNT(*) FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_NAME = '{table_name}'")).scalar()
        if exists:
            max_val = conn.execute(text(f"SELECT MAX(CAST([{date_col}] AS DATE)) FROM [{table_name}]")).scalar()
            if max_val:
                df[date_col] = pd.to_datetime(df[date_col])
                df_new = df[df[date_col] > pd.to_datetime(max_val)]
                if df_new.empty:
                    print(f" [{table_name}]: no new rows (max: {max_val})")
                    return 0
                rows_to_load = df_new
            else:
                rows_to_load = df
        else:
            rows_to_load = df
    rows_to_load.to_sql(table_name, engine, if_exists="append", index=False, chunksize=1000)
    print(f" [{table_name}]: {len(rows_to_load):,} new rows loaded")
    return len(rows_to_load)

def append_after_max_yearmonth(df, table_name):
    with engine.connect() as conn:
        exists = conn.execute(text(f"SELECT COUNT(*) FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_NAME = '{table_name}'")).scalar()
        if exists:
            result = conn.execute(text(f"SELECT MAX(Year), MAX(Month) FROM [{table_name}]")).fetchone()
            max_year, max_month = int(result[0]), int(result[1])
            if max_year and max_month:
                df_new = df[~((df['Year'] < max_year) | ((df['Year'] == max_year) & (df['Month'] <= max_month)))]
                if df_new.empty:
                    print(f"[{table_name}]: no new rows (max: {max_year}-{max_month:02d})")
                    return 0
                rows_to_load = df_new
            else:
                rows_to_load = df
        else:
            rows_to_load = df
    rows_to_load.to_sql(table_name, engine, if_exists="append", index=False, chunksize=1000)
    print(f"[{table_name}]: {len(rows_to_load):,} new rows loaded")
    return len(rows_to_load)

def replace_full_table(df, table_name):
    df.to_sql(table_name, engine, if_exists="replace", index=False, chunksize=1000)
    print(f"[{table_name}]: {len(df):,} rows replaced")
    return len(df)

# ── LOAD FILES ─────────────────────────────────────────────────────────────────

print("[1/3] Loading clean files...")
campaigns   = load_latest_clean("campaigns_clean_*.csv")
ad_groups   = load_latest_clean("ad_groups_clean_*.csv")
keywords    = load_latest_clean("keywords_clean_*.csv")
attribution = load_latest_clean("attribution_clean_*.csv")
traffic     = load_latest_clean("ga4_traffic_clean_*.csv")
users       = load_latest_clean("ga4_users_clean_*.csv")
monthly     = load_latest_clean("ga4_monthly_clean_*.csv")

# ── PUSH TO SQL SERVER ─────────────────────────────────────────────────────────

print("\n[2/3] Pushing to SQL Server...")
total_rows = 0

total_rows += append_after_max_month(campaigns, "campaigns_clean",  date_col="Month")
total_rows += append_after_max_month(ad_groups, "ad_groups_clean",  date_col="Month")

# Keywords — Month stored as string e.g. "January 2023"
with engine.connect() as conn:
    kw_exists   = conn.execute(text("SELECT COUNT(*) FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_NAME = 'keywords_clean'")).scalar()
    if kw_exists:
        kw_max_year  = conn.execute(text("SELECT MAX(Year) FROM [keywords_clean]")).scalar()
        kw_max_month = conn.execute(text(f"SELECT MAX(Month) FROM [keywords_clean] WHERE Year = {kw_max_year}")).scalar()
        if kw_max_year and kw_max_month:
            kw_max_date = pd.to_datetime(kw_max_month, format='%B %Y')
            keywords['_month_dt'] = pd.to_datetime(keywords['Month'], format='%B %Y')
            kw_new = keywords[keywords['_month_dt'] > kw_max_date].drop(columns=['_month_dt'])
            if kw_new.empty:
                print(f"[keywords_clean]: no new rows (max: {kw_max_month})")
            else:
                kw_new.to_sql("keywords_clean", engine, if_exists="append", index=False, chunksize=1000)
                print(f"[keywords_clean]: {len(kw_new):,} new rows loaded")
                total_rows += len(kw_new)
        else:
            keywords.to_sql("keywords_clean", engine, if_exists="append", index=False, chunksize=1000)
            total_rows += len(keywords)
    else:
        keywords.to_sql("keywords_clean", engine, if_exists="append", index=False, chunksize=1000)
        total_rows += len(keywords)

total_rows += replace_full_table(attribution, "Conversion_attribution_analysis")
total_rows += replace_full_table(traffic,     "Traffic_acquisition_Session_primary_channel_group_(Default_channel_group)")
total_rows += replace_full_table(users,       "User_acquisition_First_user_primary_channel_group_(Default_channel_group)")
total_rows += append_after_max_yearmonth(monthly, "Month final")

# ── VERIFY ─────────────────────────────────────────────────────────────────────

print("\n[3/3] Verifying row counts in SQL Server...")

tables = [
    "campaigns_clean", "ad_groups_clean", "keywords_clean",
    "Conversion_attribution_analysis",
    "Traffic_acquisition_Session_primary_channel_group_(Default_channel_group)",
    "User_acquisition_First_user_primary_channel_group_(Default_channel_group)",
    "Month final",
]

with engine.connect() as conn:
    for table in tables:
        try:
            count = conn.execute(text(f"SELECT COUNT(*) FROM [{table}]")).scalar()
            print(f" [{table}]: {count:,} rows total in DB")
        except Exception:
            print(f" [{table}]: table not found")

print("\n" + "=" * 60)
print("LOAD COMPLETE")
print("=" * 60)
print(f"  Rows loaded this run: {total_rows:,}")

SQL Server Load — 20260330_2153
Server:   DUARTEPC\SQLEXPRESS
Database: google_ads_analysis
 Connected to SQL Server

[1/3] Loading clean files...
 campaigns_clean_20260330_2153.csv
 ad_groups_clean_20260330_2153.csv
 keywords_clean_20260330_2153.csv
 attribution_clean_20260330_2153.csv
 ga4_traffic_clean_20260330_2153.csv
 ga4_users_clean_20260330_2153.csv
 ga4_monthly_clean_20260330_2153.csv

[2/3] Pushing to SQL Server...
 [campaigns_clean]: no new rows (max: 2026-03-01)
 [ad_groups_clean]: no new rows (max: 2026-03-01)
[keywords_clean]: no new rows (max: March 2026)
[Conversion_attribution_analysis]: 9 rows replaced
[Traffic_acquisition_Session_primary_channel_group_(Default_channel_group)]: 11 rows replaced
[User_acquisition_First_user_primary_channel_group_(Default_channel_group)]: 11 rows replaced
[Month final]: no new rows (max: 2026-12)

[3/3] Verifying row counts in SQL Server...
 [campaigns_clean]: 637 rows total in DB
 [ad_groups_clean]: 857 rows total in DB
 [keywords_clea